# 🌍 Garissa DRM - Automated Google Earth Engine Analysis

This notebook uses `geemap` and the `earthengine-api` to programmatically fetch live satellite data (Sentinel-1 Flood Extents and CHIRPS Rainfall) for Garissa County. It is designed to be run locally or via Google Cloud Vertex AI Notebooks.

In [ ]:
!pip install geemap earthengine-api pandas geopandas --quiet

In [ ]:
import ee
import geemap

# Authenticate and Initialize Earth Engine
# If running locally for the first time, this will open a browser window for authentication.
try:
    ee.Initialize(project='garissadrm')
    print("✅ Successfully connected to Google Earth Engine Project: 'garissadrm'")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='garissadrm')

In [ ]:
# 1. Define Garissa Region of Interest (ROI)
garissa = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(ee.Filter.eq('ADM1_NAME', 'Garissa'))

# 2. Setup the Map
Map = geemap.Map()
Map.centerObject(garissa, 8)
Map.add_basemap('HYBRID')

# Add Garissa boundary
empty = ee.Image().byte()
outline = empty.paint(garissa, 1, 3)
Map.addLayer(outline, {'palette': ['00FFFF']}, 'Garissa Boundary')

Map

In [ ]:
# 3. Fetch Sentinel-1 Live Flood Radar (Last 30 Days)
today = ee.Date(ee.Date.now())
last_month = today.advance(-30, 'day')

sentinel1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterBounds(garissa) \
    .filterDate(last_month, today) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .select('VV') \
    .mosaic() \
    .clip(garissa)

# Calculate water mask (values less than -16 dB are typically water in VV polarization)
water_mask = sentinel1.lt(-16)
water_layer = water_mask.updateMask(water_mask)

Map.addLayer(water_layer, {'min': 0, 'max': 1, 'palette': ['FF007F']}, 'Live Floods (Sentinel-1)')
print("✅ Added Sentinel-1 Flood Layer to the Map")

In [ ]:
# 4. Calculate total flooded area in square kilometers

# Calculate pixel area in square meters
area_img = water_layer.multiply(ee.Image.pixelArea())

# Sum the area across Garissa
stats = area_img.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=garissa.geometry(),
    scale=100,
    maxPixels=1e9
)

total_area_sqm = stats.get('VV').getInfo()
total_area_sqkm = total_area_sqm / 1e6

print(f"⚠️ Current Flooded Area in Garissa: {total_area_sqkm:.2f} sq km")